# Game-region renderer preview

Run sections independently after the setup cell. These are game panels only, with no desktop, window chrome, augmentation, or export. Thumbnails fit large boards on screen; the optional native-resolution header crop shows control details. Seeds and examples can be edited below. The Microsoft toolbar is a coherent procedural approximation, not a reconstruction of a particular release.


In [ ]:
from IPython.display import display
from PIL import ImageDraw
from synthetic_scenes.minesweeper.board import generate_game
from synthetic_scenes.minesweeper.renderer import render_game_region


def preview(style, rows, columns, mines, cell_size, theme="default",
            status="active", seed=42, variant="classic", negative=False, annotations=False):
    game = generate_game(rows, columns, mines, status=status,
                         progress=0.35 if status == "active" else 0,
                         seed=seed, elapsed_seconds=0 if status == "new" else 137)
    if negative:
        # Exercise a valid over-flagged field without changing the minefield.
        game["flags"][:] = False
        game["flags"].flat[:min(rows * columns, mines + 15)] = True
    image, metadata = render_game_region(game, style=style, theme=theme,
                                         cell_size=cell_size, mine_variant=variant)
    if annotations:
        image = image.copy()
        draw = ImageDraw.Draw(image)
        colors = {"game_region": "red", "grid": "cyan", "mine_counter": "yellow",
                  "timer": "magenta", "reset_button": "lime"}
        for name, (x1, y1, x2, y2) in metadata["boxes"].items():
            draw.rectangle((x1, y1, x2-1, y2-1), outline=colors[name], width=2)
            draw.text((x1+2, y1+2), name, fill=colors[name])
    print(f"{style}: {rows}x{columns}, {theme}, {status}, cell {cell_size}px; "
          f"panel {image.size}, boxes {metadata['boxes']}")
    thumbnail = image.copy()
    thumbnail.thumbnail((1100, 700))
    display(thumbnail)
    # Inspect the panel controls at native size; Win7 controls are at the bottom.
    boxes = [metadata['boxes'][key] for key in ('mine_counter', 'timer')]
    y1 = max(0, min(box[1] for box in boxes) - cell_size // 2)
    y2 = min(image.height, max(box[3] for box in boxes) + cell_size // 2)
    display(image.crop((0, y1, image.width, y2)))


## Classic

Beginner, intermediate and expert sizes, the four themes, each game state, and a negative counter.


In [ ]:
preview("classic", 9, 9, 10, 16, status="new")
preview("classic", 16, 16, 40, 24, theme="blue", status="active")
preview("classic", 16, 30, 99, 12, theme="green", status="won")
preview("classic", 5, 5, 4, 48, theme="warm", status="lost", negative=True)


## Windows 7

Blue and green glass, bottom controls, and both mine variants.


In [ ]:
preview("windows7", 9, 9, 10, 32, status="new")
preview("windows7", 16, 16, 40, 16, theme="blue", status="active")
preview("windows7", 16, 30, 99, 12, theme="green", status="won", variant="flower")
preview("windows7", 20, 5, 15, 24, status="lost")


## Microsoft

The header, typography and control placement are approximate; the existing tile style is unchanged.


In [ ]:
preview("microsoft", 9, 9, 10, 24, status="new")
preview("microsoft", 16, 16, 40, 16, theme="blue", status="active")
preview("microsoft", 16, 30, 99, 12, theme="warm", status="lost")
preview("microsoft", 5, 20, 15, 32, status="won")


## Google

The custom layouts include narrow and wide boards and all toolbar colors.


In [ ]:
preview("google", 8, 10, 10, 32, status="new")
preview("google", 14, 18, 40, 24, theme="blue", status="active")
preview("google", 20, 24, 99, 16, theme="warm", status="won")
preview("google", 20, 5, 20, 64, status="lost")


## Large boards and counters

High mine counts use extra slots. Negative examples simulate over-flagging; the board remains unchanged.


In [ ]:
preview("classic", 100, 100, 3500, 12, status="new")
preview("classic", 50, 50, 100, 12, status="new", negative=True)
preview("google", 30, 100, 1200, 12, status="new")


## Annotation alignment

Red: panel; cyan: grid; yellow: mine counter; magenta: timer; green: reset. Boxes use half-open coordinates.


In [ ]:
for style in ("classic", "windows7", "microsoft", "google"):
    preview(style, 9, 5, 10, 24, status="active", annotations=True)
